In [142]:
import pgmpy as pg
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD



In [143]:
model = DiscreteBayesianNetwork(
    [
        ('DebtIncomeRatio', 'PaymentHistory'),
        ('PaymentHistory', 'Age'),
        ('PaymentHistory', 'Reliability'),
        ('Age', 'Reliability'),
        ('Income','Assets'), 
        ('Income', 'FutureIncome'),
        ('Assets', 'FutureIncome'),
        ('DebtIncomeRatio', 'BankLoan'),
        ('Reliability', 'BankLoan'),
        ('FutureIncome', 'BankLoan')
    ]
)

In [144]:
model.to_graphviz().draw('tp2.png',prog='dot')

In [145]:
# Define CPD for DebtIncomeRatio (no parents)
cpd_debt = TabularCPD('DebtIncomeRatio', 2,
                      [[0.4], [0.6]])

# Define CPD for PaymentHistory given DebtIncomeRatio
# Observation 4: High debt ratio leads to worse payment history
cpd_payment = TabularCPD('PaymentHistory', 3,
                        [[0.8, 0.2],    # Excellent
                         [0.15, 0.3],   # Acceptable
                         [0.05, 0.5]],  # Unacceptable
                        evidence=['DebtIncomeRatio'],
                        evidence_card=[2])

# Define CPD for Age given PaymentHistory
# Observation 3: Better payment history correlates with older age
cpd_age = TabularCPD('Age', 3,
                    [[0.6, 0.3, 0.1],    # Between16and25
                     [0.3, 0.4, 0.3],    # Between25and40
                     [0.1, 0.3, 0.6]],   # Over65
                    evidence=['PaymentHistory'],
                    evidence_card=[3])

# Define CPD for Reliability given PaymentHistory and Age
# Observation 1 & 2: Both good payment history and older age increase reliability
cpd_reliability = TabularCPD('Reliability', 2,
                            [[0.9, 0.6, 0.1, 0.7, 0.4, 0.05, 0.8, 0.5, 0.15],
                             [0.1, 0.4, 0.9, 0.3, 0.6, 0.95, 0.2, 0.5, 0.85]],
                            evidence=['PaymentHistory', 'Age'],
                            evidence_card=[3, 3])

# Define CPD for Income (no parents)
cpd_income = TabularCPD('Income', 2,
                       [[0.3], [0.7]])

# Define CPD for Assets given Income
# Observation 5: Higher income correlates with more assets
cpd_assets = TabularCPD('Assets', 2,
                       [[0.8, 0.3],    # Low
                        [0.2, 0.7]],   # High
                       evidence=['Income'],
                       evidence_card=[2])

# Define CPD for FutureIncome given Income and Assets
# Observation 6: High income and assets increase future income
cpd_future_income = TabularCPD('FutureIncome', 2,
                              [[0.85, 0.5, 0.4, 0.15],    # Low
                               [0.15, 0.5, 0.6, 0.85]],   # High
                              evidence=['Income', 'Assets'],
                              evidence_card=[2, 2])

# Define CPD for BankLoan given Reliability, FutureIncome, and DebtIncomeRatio
# Observation 7: Reliable persons, with good future income, and low debt ratio get loans
cpd_loan = TabularCPD('BankLoan', 2,
                     [[0.95, 0.7, 0.5, 0.2, 0.8, 0.5, 0.3, 0.05],    # No
                      [0.05, 0.3, 0.5, 0.8, 0.2, 0.5, 0.7, 0.95]],   # Yes
                     evidence=['Reliability', 'FutureIncome', 'DebtIncomeRatio'],
                     evidence_card=[2, 2, 2])

# Add all CPDs to the model
model.add_cpds(cpd_debt, cpd_payment, cpd_age, cpd_reliability,
              cpd_income, cpd_assets, cpd_future_income, cpd_loan)

# Verify the model is valid
print("Model valid:", model.check_model())
print("\nAll CPDs added successfully!")

Model valid: True

All CPDs added successfully!


## Section 2: Local Independence Analysis

Using pgmpy's capabilities to find conditional independencies between variables.

In [146]:
# Get local independencies
print("Local Independencies in the network:")
print(model.get_independencies())

# Check independencies for Income (using d-separation)
print("\n" + "="*50)
print("Variables independent of Income:")
print("="*50)
for node in model.nodes():
    if node != 'Income':
        if not model.is_dconnected('Income', node):
            print(f"  {node} is independent of Income")

# Check active trails from Income
print("\nActive trails from Income:")
for node in model.nodes():
    if node != 'Income':
        if model.is_dconnected('Income', node):
            print(f"  Income ---> {node} (d-connected)")

# Check independencies given BankLoan
print("\n" + "="*50)
print("Variables independent of Income given BankLoan:")
print("="*50)
for node in model.nodes():
    if node != 'Income':
        if model.is_dconnected('Income', node, observed=['BankLoan']):
            print(f"  Income and {node} are d-connected given BankLoan")
        else:
            print(f"  Income and {node} are d-separated given BankLoan (INDEPENDENT)")

Local Independencies in the network:
(Assets ⟂ DebtIncomeRatio)
(Age ⟂ Income)
(Assets ⟂ BankLoan | FutureIncome)
(Age ⟂ DebtIncomeRatio | PaymentHistory)
(PaymentHistory ⟂ FutureIncome)
(PaymentHistory ⟂ Assets)
(Reliability ⟂ Assets)
(Reliability ⟂ FutureIncome)
(Reliability ⟂ DebtIncomeRatio | PaymentHistory)
(Reliability ⟂ Income)
(Age ⟂ FutureIncome)
(DebtIncomeRatio ⟂ Income)
(FutureIncome ⟂ DebtIncomeRatio)
(Age ⟂ Assets)
(PaymentHistory ⟂ BankLoan | DebtIncomeRatio, Reliability)
(PaymentHistory ⟂ Income)
(BankLoan ⟂ Income | FutureIncome)
(Age ⟂ BankLoan | DebtIncomeRatio, Reliability)

Variables independent of Income:
  DebtIncomeRatio is independent of Income
  PaymentHistory is independent of Income
  Age is independent of Income
  Reliability is independent of Income

Active trails from Income:
  Income ---> Assets (d-connected)
  Income ---> FutureIncome (d-connected)
  Income ---> BankLoan (d-connected)

Variables independent of Income given BankLoan:
  Income and DebtInc

## Section 3: Learning from Data

Learn CPD distributions from the 50000-cases.csv file using Maximum Likelihood Estimation.

In [147]:
import pandas as pd
from pgmpy.estimators import MaximumLikelihoodEstimator

# Load data
data = pd.read_csv('50000-cases.csv')
print("Data shape:", data.shape)
#print("\nFirst few rows:")
print(data.head())
#print("\nData info:")
#print(data.info())
print(data.dtypes)
# Create a new model with the same structure but learn CPDs from data
model_learned = DiscreteBayesianNetwork(
    [
        ('DebtIncomeRatio', 'PaymentHistory'),
        ('PaymentHistory', 'Age'),
        ('PaymentHistory', 'Reliability'),
        ('Age', 'Reliability'),
        ('Income','Assets'), 
        ('Income', 'FutureIncome'),
        ('Assets', 'FutureIncome'),
        ('DebtIncomeRatio', 'BankLoan'),
        ('Reliability', 'BankLoan'),
        ('FutureIncome', 'BankLoan')
    ]
)

# Fit CPDs using Maximum Likelihood Estimation
mle = MaximumLikelihoodEstimator(model_learned, data)
model_learned.fit(data, estimator=MaximumLikelihoodEstimator)

print("\n" + "="*50)
print("Learned CPDs from data:")
print("="*50)
for cpd in model_learned.get_cpds():
    print(f"\n{cpd}")

INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'DebtIncomeRatio': 'C', 'Income': 'C', 'PaymentHistory': 'C', 'Assets': 'C', 'Age': 'C', 'FutureIncome': 'C', 'Reliability': 'C', 'BankLoan': 'C'}
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'DebtIncomeRatio': 'C', 'Income': 'C', 'PaymentHistory': 'C', 'Assets': 'C', 'Age': 'C', 'FutureIncome': 'C', 'Reliability': 'C', 'BankLoan': 'C'}


Data shape: (50000, 8)
  DebtIncomeRatio  Income PaymentHistory  Assets             Age  \
0             Low  Medium      Excellent  Medium  Between26and64   
1             Low    High      Excellent    High          Over65   
2             Low  Medium   Unacceptable    High  Between26and64   
3            High  Medium   Unacceptable     Low          Over65   
4            High  Medium     Acceptable  Medium          Over65   

    FutureIncome Reliability  BankLoan  
0      Promising    Reliable  Negative  
1      Promising    Reliable  Positive  
2      Promising  Unreliable  Positive  
3  Not_promising    Reliable  Positive  
4      Promising  Unreliable  Negative  
DebtIncomeRatio    object
Income             object
PaymentHistory     object
Assets             object
Age                object
FutureIncome       object
Reliability        object
BankLoan           object
dtype: object

Learned CPDs from data:

+-----------------------+---------+
| DebtIncomeRatio(High) | 0.50138 |
+-

## Section 4: Exact Inference

Calculate probabilities using Variable Elimination.

In [148]:
from pgmpy.inference import VariableElimination
import time

# Use the learned model for inference
infer = VariableElimination(model_learned)

print("="*60)
print("EXACT INFERENCE USING VARIABLE ELIMINATION")
print("="*60)

# Query 1: P(BankLoan)
print("\n1. P(BankLoan)")
start = time.time()
result1 = infer.query(variables=['BankLoan'])
time1 = time.time() - start
print(result1)
print(f"Time: {time1:.4f}s")

# Query 2: P(BankLoan | Income=Low, Age=Between16and25, PaymentHistory=Excellent, Assets=Low)
print("\n2. P(BankLoan | Income=Low, Age=Between16and25, PaymentHistory=Excellent, Assets=Low)")
start = time.time()
result2 = infer.query(variables=['BankLoan'], 
                     evidence={'Income': 'Low', 'Age': 'Between16and25', 
                              'PaymentHistory': 'Excellent', 'Assets': 'Low'})
time2 = time.time() - start
print(result2)
print(f"Time: {time2:.4f}s")

# Query 3: P(BankLoan | Income=High, Age=Between16and25, PaymentHistory=Excellent, Assets=High)
print("\n3. P(BankLoan | Income=High, Age=Between16and25, PaymentHistory=Excellent, Assets=High)")
start = time.time()
result3 = infer.query(variables=['BankLoan'], 
                     evidence={'Income': 'High', 'Age': 'Between16and25', 
                              'PaymentHistory': 'Excellent', 'Assets': 'High'})
time3 = time.time() - start
print(result3)
print(f"Time: {time3:.4f}s")

# Query 4: P(BankLoan | Income=High, Age=Over65, PaymentHistory=Excellent, Assets=High)
print("\n4. P(BankLoan | Income=High, Age=Over65, PaymentHistory=Excellent, Assets=High)")
start = time.time()
result4 = infer.query(variables=['BankLoan'], 
                     evidence={'Income': 'High', 'Age': 'Over65', 
                              'PaymentHistory': 'Excellent', 'Assets': 'High'})
time4 = time.time() - start
print(result4)
print(f"Time: {time4:.4f}s")

EXACT INFERENCE USING VARIABLE ELIMINATION

1. P(BankLoan)
+--------------------+-----------------+
| BankLoan           |   phi(BankLoan) |
+====================+=================+
| BankLoan(Negative) |          0.4574 |
+--------------------+-----------------+
| BankLoan(Positive) |          0.5426 |
+--------------------+-----------------+
Time: 0.0031s

2. P(BankLoan | Income=Low, Age=Between16and25, PaymentHistory=Excellent, Assets=Low)
+--------------------+-----------------+
| BankLoan           |   phi(BankLoan) |
+====================+=================+
| BankLoan(Negative) |          0.4674 |
+--------------------+-----------------+
| BankLoan(Positive) |          0.5326 |
+--------------------+-----------------+
Time: 0.0019s

3. P(BankLoan | Income=High, Age=Between16and25, PaymentHistory=Excellent, Assets=High)
+--------------------+-----------------+
| BankLoan           |   phi(BankLoan) |
+====================+=================+
| BankLoan(Negative) |          0.3065 |

## Section 5: Inference by Forward Sampling

Estimate the same probabilities using forward sampling and compare with exact inference.

In [ ]:
import time
from pgmpy.sampling import BayesianModelSampling

# Initialize the sampler
sampler = BayesianModelSampling(model_learned)

print("="*60)
print("APPROXIMATE INFERENCE (REJECTION SAMPLING)")
print("="*60)
print("(Using 10,000 samples)")

# Query 1: P(BankLoan) - No evidence, forward sampling is efficient
print("\n1. P(BankLoan) - Approximate")
start = time.time()
samples1 = sampler.forward_sample(size=10000)
time1_approx = time.time() - start
print(samples1['BankLoan'].value_counts(normalize=True))
print(f"Time: {time1_approx:.4f}s")

# Query 2: P(BankLoan | Income=Low, Age=Between16and25, PaymentHistory=Excellent, Assets=Low)
print("\n2. P(BankLoan | Income=Low, Age=Between16and25, PaymentHistory=Excellent, Assets=Low) - Approximate")
evidence2 = [('Income', 'Low'), ('Age', 'Between16and25'), 
             ('PaymentHistory', 'Excellent'), ('Assets', 'Low')]
start = time.time()
samples2 = sampler.rejection_sample(evidence=evidence2, size=10000)
time2_approx = time.time() - start
print(samples2['BankLoan'].value_counts(normalize=True))
print(f"Time: {time2_approx:.4f}s")

# Query 3: P(BankLoan | Income=High, Age=Between16and25, PaymentHistory=Excellent, Assets=High)
print("\n3. P(BankLoan | Income=High, Age=Between16and25, PaymentHistory=Excellent, Assets=High) - Approximate")
evidence3 = [('Income', 'High'), ('Age', 'Between16and25'), 
             ('PaymentHistory', 'Excellent'), ('Assets', 'High')]
start = time.time()
samples3 = sampler.rejection_sample(evidence=evidence3, size=10000)
time3_approx = time.time() - start
print(samples3['BankLoan'].value_counts(normalize=True))
print(f"Time: {time3_approx:.4f}s")

# Query 4: P(BankLoan | Income=High, Age=Over65, PaymentHistory=Excellent, Assets=High)
print("\n4. P(BankLoan | Income=High, Age=Over65, PaymentHistory=Excellent, Assets=High) - Approximate")
evidence4 = [('Income', 'High'), ('Age', 'Over65'), 
             ('PaymentHistory', 'Excellent'), ('Assets', 'High')]
start = time.time()
samples4 = sampler.rejection_sample(evidence=evidence4, size=10000)
time4_approx = time.time() - start
print(samples4['BankLoan'].value_counts(normalize=True))
print(f"Time: {time4_approx:.4f}s")

print("\n" + "="*60)
print("COMPARISON: Exact vs Approximate Inference")
print("="*60)
# Note: Ensure variables time1, time2, etc., from your exact inference script are defined in your environment
print(f"Query 1 - Exact: {time1:.4f}s | Approximate: {time1_approx:.4f}s")
print(f"Query 2 - Exact: {time2:.4f}s | Approximate: {time2_approx:.4f}s")
print(f"Query 3 - Exact: {time3:.4f}s | Approximate: {time3_approx:.4f}s")
print(f"Query 4 - Exact: {time4:.4f}s | Approximate: {time4_approx:.4f}s")

APPROXIMATE INFERENCE (REJECTION SAMPLING)
(Using 10,000 samples)

1. P(BankLoan) - Approximate


  0%|          | 0/8 [00:00<?, ?it/s]

BankLoan
Positive    0.5429
Negative    0.4571
Name: proportion, dtype: float64
Time: 0.1576s

2. P(BankLoan | Income=Low, Age=Between16and25, PaymentHistory=Excellent, Assets=Low) - Approximate


  0%|          | 0/10000 [00:00<?, ?it/s]

BankLoan
Positive    0.5401
Negative    0.4599
Name: proportion, dtype: float64
Time: 13.8925s

3. P(BankLoan | Income=High, Age=Between16and25, PaymentHistory=Excellent, Assets=High) - Approximate


  0%|          | 0/10000 [00:00<?, ?it/s]

BankLoan
Positive    0.6966
Negative    0.3034
Name: proportion, dtype: float64
Time: 7.8891s

4. P(BankLoan | Income=High, Age=Over65, PaymentHistory=Excellent, Assets=High) - Approximate


  0%|          | 0/10000 [00:00<?, ?it/s]

BankLoan
Positive    0.7354
Negative    0.2646
Name: proportion, dtype: float64
Time: 1.4052s

COMPARISON: Exact vs Approximate Inference
Query 1 - Exact: 0.0031s | Approximate: 0.1576s
Query 2 - Exact: 0.0019s | Approximate: 13.8925s
Query 3 - Exact: 0.0022s | Approximate: 7.8891s
Query 4 - Exact: 0.0020s | Approximate: 1.4052s
